# Vehicle detector fine-tuning on PKLot (Component 1, hypothesis H4)

Fine-tunes COCO-pretrained YOLO11 and the YOLOv8 baseline to detect `vehicle` in high-angle car-park views.

* **Data:** `pklot_yolo.zip`, made by `eval/prepare_pklot.py convert` (day-level split; UFPR05 held out; images masked outside annotated spaces; weak boxes from occupied-space outlines).
* **Run on:** Kaggle, Accelerator = GPU T4 x1 (or P100), Internet = on. Add the uploaded dataset with *Add Input*.
* **Outputs** (download from the Output tab): `runs/<name>/weights/best.pt`, `results.csv`, `args.yaml`, and `run_summary.json` with the weights' SHA-256.

The validation mAP printed here is measured against the **weak** labels, so it is only a training signal. Report detection accuracy from `eval_detection.py` on the hand-verified gold set.

In [ ]:
!pip install -q ultralytics==8.4.178
import ultralytics, torch
print(ultralytics.__version__, torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
import os, zipfile
from pathlib import Path

def find_roots(base):
    """Folders holding images/ (os.walk follows Kaggle's symlinked mounts, Path.rglob does not)."""
    return [Path(d) for d, dirs, _ in os.walk(base, followlinks=True) if 'images' in dirs]

roots = find_roots('/kaggle/input')
if not roots:
    # Kaggle sometimes keeps the upload as a .zip: extract it to the writable working folder.
    zips = [Path(d) / f for d, _, fs in os.walk('/kaggle/input', followlinks=True) for f in fs if f.endswith('.zip')]
    assert zips, 'No dataset found: in the right panel use Input > + Add Input and add your PKLot dataset.'
    print('Extracting', zips[0])
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working/data')
    roots = find_roots('/kaggle/working/data')

for r in roots:
    print(r, '->', sorted(os.listdir(r)), '| labels/train present:', (r / 'labels' / 'train').is_dir())
good = [r for r in roots if (r / 'images' / 'train').is_dir() and (r / 'labels' / 'train').is_dir()]
assert good, 'images/ found but labels/ is missing: the upload is incomplete (see the lines above).'
DATA_ROOT = good[0]
yaml = Path('/kaggle/working/pklot_vehicle.yaml')
yaml.write_text(f'path: {DATA_ROOT}
train: images/train
val: images/val
names:
  0: vehicle
')
print('DATA_ROOT =', DATA_ROOT)
for split in ('train', 'val'):
    n_img = len(list((DATA_ROOT / 'images' / split).glob('*.jpg')))
    n_lbl = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split}: {n_img} images, {n_lbl} label files')

In [ ]:
import hashlib, json, time
from ultralytics import YOLO

# Identical settings for every model so the comparison is fair. Ultralytics default augmentation is used
# (HSV, flips, mosaic, scale); it applies to training images only, never to val/test.
COMMON = dict(data=str(yaml), imgsz=1280, epochs=50, patience=15, batch=8, seed=0, deterministic=True,
              cos_lr=True, workers=4, project='/kaggle/working/runs', exist_ok=True, plots=True)

def train(weights: str, name: str) -> dict:
    t = time.time()
    model = YOLO(weights)
    # Safety net only: stop after the epoch that passes 10.5 h so best.pt is saved before Kaggle's 12-hour hard
    # limit. (Ultralytics' own `time=` argument is not used because it re-sets the epoch count to fill the time.)
    def stop_before_limit(trainer, t0=t, limit_h=10.5):
        if time.time() - t0 > limit_h * 3600:
            trainer.stop = True
    model.add_callback('on_fit_epoch_end', stop_before_limit)
    model.train(name=name, **COMMON)
    best = Path(COMMON['project']) / name / 'weights' / 'best.pt'
    summary = {'name': name, 'base_weights': weights, 'settings': {k: v for k, v in COMMON.items() if k != 'data'},
               'best_pt_sha256': hashlib.sha256(best.read_bytes()).hexdigest(),
               'train_hours': round((time.time() - t) / 3600, 2),
               'gpu': torch.cuda.get_device_name(0), 'ultralytics': ultralytics.__version__,
               'note': 'val metrics are against weak labels; report gold-set metrics instead'}
    (best.parent.parent / 'run_summary.json').write_text(json.dumps(summary, indent=1))
    print(json.dumps(summary, indent=1))
    return summary

In [ ]:
# >>> CHANGE ONLY THIS LINE between runs <<<
# Run 1: 'yolo11s.pt'   Run 2: 'yolov8s.pt'   Optional run 3 (speed comparison, NFR-01): 'yolo11n.pt'
MODEL = 'yolo11s.pt'

# About 3-4 hours per model on a T4. One model per run keeps each run well inside Kaggle's 12-hour limit.
summary = train(MODEL, f"pklot_{MODEL.removesuffix('.pt')}_1280")

In [ ]:
# Bundle the training results to download in one file (weights are large; keep them out of git).
!cd /kaggle/working && zip -qr pklot_runs.zip runs && ls -lh pklot_runs.zip